# Video Analytics with DL Streamer (V1)

Minimal walkthrough: **GStreamer** carries the media path; **DL Streamer** plugins add detection, tracking, and classification; **OpenVINO** runs inference under those elements.

**Use case:** detect cars and classify **color** per car on sample traffic video.

**Run on:** Linux host with DL Streamer installed (paths below target the demo server). Editing this repo on Windows does not require local DL Streamer.

## Pipeline flow (what each stage does)

| Stage | GStreamer / DL Streamer |
|--------|-------------------------|
| **Decode** | `filesrc` → `decodebin3` |
| **Pre-process** | `videoconvert`; elements use OpenCV preproc where noted (`pre-process-backend=opencv`) |
| **Inference** | `gvadetect` (cars), `gvaclassify` (color) — OpenVINO under the hood |
| **Post-process** | `gvatrack`, `gvawatermark`, `gvafpscounter` |
| **Output** | `autovideosink` (live demo) or `fakesink` (throughput benchmarks) |

Progression in this notebook: raw video → +FPS → +detection → +classification → multi-stream `fakesink` benchmarks.

## Precision and devices

- **Precision** = which `model.xml` you select (`FP16` / `FP32` / `INT8` under `car-model/`). Default: **FP16**.
- **Devices** = OpenVINO device passed to elements: detection **`device=$DETECTION_DEVICE`**, classification **`device=$CLASSIFICATION_DEVICE`** (defaults: GPU + NPU).
- Change defaults in `utils.py` or override env vars after `apply_environment()`.

### 1. Configure paths (Python + `utils.py`)

Loads server paths and exports them into the kernel environment so **`%%bash`** cells can use `$VIDEO_SRC`, `$DETECTION_MODEL`, etc.

In [ ]:
import sys
from pathlib import Path

for d in (Path.cwd(), Path.cwd() / "2"):
    if (d / "utils.py").is_file():
        if str(d) not in sys.path:
            sys.path.insert(0, str(d))
        break

from utils import apply_environment, validate_default_assets

apply_environment()
try:
    validate_default_assets()
    print("Assets found.")
except FileNotFoundError as e:
    print("Validation skipped or failed (normal when not on the demo server):", e)

import os
for k in (
    "VIDEO_SRC",
    "DETECTION_MODEL",
    "CLASSIFICATION_MODEL",
    "DETECTION_DEVICE",
    "CLASSIFICATION_DEVICE",
    "PRECISION",
):
    print(f"{k}={os.environ.get(k)}")

### 2. Show the video

Close the video window when done — that is expected. Cells use `set +e` so exit codes stay friendly.

In [ ]:
%%bash
set +e

gst-launch-1.0 \
  filesrc location="$VIDEO_SRC" ! \
  decodebin3 ! \
  videoconvert ! \
  autovideosink sync=true

echo "Pipeline ended."

### 3. Video + FPS counter

In [ ]:
%%bash
set +e

gst-launch-1.0 \
  filesrc location="$VIDEO_SRC" ! \
  decodebin3 ! \
  videoconvert ! \
  gvafpscounter ! \
  autovideosink sync=true

echo "Pipeline ended."

### 4. Add detection (cars)

In [ ]:
%%bash
set +e

gst-launch-1.0 \
  filesrc location="$VIDEO_SRC" ! \
  decodebin3 ! \
  gvadetect model="$DETECTION_MODEL" device="$DETECTION_DEVICE" pre-process-backend=opencv ! \
  gvawatermark ! \
  gvafpscounter ! \
  videoconvert ! \
  autovideosink sync=true

echo "Pipeline ended."

### 5. Detection + classification (car color)

`gvatrack` keeps boxes stable; `gvaclassify` runs the color head on crops.

In [ ]:
%%bash
set +e

gst-launch-1.0 \
  filesrc location="$VIDEO_SRC" ! \
  decodebin3 ! \
  gvadetect model="$DETECTION_MODEL" device="$DETECTION_DEVICE" pre-process-backend=opencv ! \
  gvatrack ! \
  gvaclassify model="$CLASSIFICATION_MODEL" device="$CLASSIFICATION_DEVICE" pre-process-backend=opencv reclassify-interval=2 ! \
  queue ! \
  gvawatermark ! \
  gvafpscounter ! \
  videoconvert ! \
  autovideosink sync=true

echo "Pipeline ended."

### 6. Benchmark: 2 parallel streams (`fakesink`)

Strict shell mode for non-interactive runs.

In [ ]:
%%bash
set -euo pipefail

gst-launch-1.0 \
  filesrc location="$VIDEO_SRC" ! decodebin3 ! \
  gvadetect model="$DETECTION_MODEL" device="$DETECTION_DEVICE" pre-process-backend=opencv ! \
  gvatrack ! \
  gvaclassify model="$CLASSIFICATION_MODEL" device="$CLASSIFICATION_DEVICE" pre-process-backend=opencv reclassify-interval=2 ! \
  queue ! gvafpscounter ! fakesink sync=false \
  filesrc location="$VIDEO_SRC" ! decodebin3 ! \
  gvadetect model="$DETECTION_MODEL" device="$DETECTION_DEVICE" pre-process-backend=opencv ! \
  gvatrack ! \
  gvaclassify model="$CLASSIFICATION_MODEL" device="$CLASSIFICATION_DEVICE" pre-process-backend=opencv reclassify-interval=2 ! \
  queue ! gvafpscounter ! fakesink sync=false

### 7. Benchmark: 4 parallel streams (`fakesink`)

In [ ]:
%%bash
set -euo pipefail

gst-launch-1.0 \
  filesrc location="$VIDEO_SRC" ! decodebin3 ! gvadetect model="$DETECTION_MODEL" device="$DETECTION_DEVICE" pre-process-backend=opencv ! gvatrack ! gvaclassify model="$CLASSIFICATION_MODEL" device="$CLASSIFICATION_DEVICE" pre-process-backend=opencv reclassify-interval=2 ! queue ! gvafpscounter ! fakesink sync=false \
  filesrc location="$VIDEO_SRC" ! decodebin3 ! gvadetect model="$DETECTION_MODEL" device="$DETECTION_DEVICE" pre-process-backend=opencv ! gvatrack ! gvaclassify model="$CLASSIFICATION_MODEL" device="$CLASSIFICATION_DEVICE" pre-process-backend=opencv reclassify-interval=2 ! queue ! gvafpscounter ! fakesink sync=false \
  filesrc location="$VIDEO_SRC" ! decodebin3 ! gvadetect model="$DETECTION_MODEL" device="$DETECTION_DEVICE" pre-process-backend=opencv ! gvatrack ! gvaclassify model="$CLASSIFICATION_MODEL" device="$CLASSIFICATION_DEVICE" pre-process-backend=opencv reclassify-interval=2 ! queue ! gvafpscounter ! fakesink sync=false \
  filesrc location="$VIDEO_SRC" ! decodebin3 ! gvadetect model="$DETECTION_MODEL" device="$DETECTION_DEVICE" pre-process-backend=opencv ! gvatrack ! gvaclassify model="$CLASSIFICATION_MODEL" device="$CLASSIFICATION_DEVICE" pre-process-backend=opencv reclassify-interval=2 ! queue ! gvafpscounter ! fakesink sync=false

## Summary

- **V1** is a clean, linear demo: decode → optional FPS → detect → classify → multi-stream throughput.
- **Config** lives in `utils.py`; **core `gst-launch` graphs** stay in the cells above.
- **Visual** cells: `set +e` + `autovideosink`. **Benchmark** cells: `set -euo pipefail` + `fakesink`.
- To switch **FP32** / **INT8**, change `DEFAULT_PRECISION` in `utils.py` or call `apply_environment(precision="INT8")` before bash cells.